# Orange County Analysis

In [ ]:
# Import statements
import pandas as pd
import numpy as np

# Display all columns
pd.set_option('display.max_columns', None)

# Import Data

In [ ]:
# Load in policing data
original_policing = pd.read_csv("../data/cleaned/cleaned_orange_ripa_2022_2024.csv", low_memory=False)

# Select only relevant columns to use
keep_cols = [
    "doj_record_id",
    "person_number",
    "agency_ori",
    "agency_name",
    "time_of_stop",
    "date_of_stop",
    "year",
    "stop_duration",
    "closest_city",
    "race_ethnicity",
    "gender",
    "age",
    "reason_for_contact",
    "traffic_violation_cjis_offense_code",
    "suspicion_cjis_offense_code",
    "suspicion",
    "action_any_search",
    "search_basis_plain_view",
    "search_basis_plain_smell",
    "search_basis_consent",
    "search_basis_safety",
    "search_basis_suspect_weapon",
    "search_basis_evidence_of_crime",
    "search_basis_school_policy",
    "search_basis_emergency",
    "search_basis_canine",
    "search_basis_warrant",
    "search_basis_probation",
    "search_basis_incident_to_arrest",
    "search_basis_vehicle_inventory",
    "contraband_any",
    "result_of_stop_arrest",
    "search_type",
    "multi_person_stop"
]
policing = original_policing[[c for c in keep_cols if c in original_policing.columns]].copy()

In [ ]:
# Load in prosecution data
prosecution = pd.read_csv("https://raw.githubusercontent.com/laurenbchu/honors-thesis/main/data/cleaned/cleaned_orange_aclu_2021_2023.csv", low_memory=False)
prosecution['filed_date'] = pd.to_datetime(prosecution["filed_date"])
prosecution['Year'] = prosecution['filed_date'].dt.year

In [ ]:
from rates_utils import load_census, census_rollup

# Load in census data
census = load_census("06059")

# Roll-up census data to be more coarse to match policing/prosecution race categories
census_coarse = census_rollup(census)

# Policing Analysis

In [ ]:
from rates_utils import add_standardized_race, policing_rates

# Filter for only discretionary stops
policing = policing[policing["reason_for_contact"].isin([
    "Moving violation",
    "Equipment violation",
    "Non-moving violation",
    "Suspect criminal activity"
])].copy()

# Add standardized race column to match policing and prosecution labels
policing = add_standardized_race(policing, "race_ethnicity")

# Consider only purely discretionary searches in search and hit rates

# Clean contraband_any (handles NaN, 0/1, True/False)
policing["contraband_any"] = policing["contraband_any"].fillna(0).astype(int)

# Create discretionary search indicator
policing["disc_search"] = policing["action_any_search"] & policing["search_type"].eq("Discretionary only")

# Create hits among discretionary searches indicator
# Hits only count as hits if they were found in a search, and the search was discretionary
policing["disc_hit"] = (
    policing["disc_search"] & # Has to be from a discretionary search
    policing["action_any_search"].fillna(False) & # Has to have been searched
    (policing["contraband_any"] == 1)
)

# Create a copy with all people involved in stops for sensitivity analysis
policing_multi = policing.copy()

# Filter for person 1 for each stop (usually, the main person stopped, but not always)
policing = policing[policing["person_number"] == 1].copy()

policing_analysis = policing_rates(policing, census_coarse)

In [ ]:
# Creating tables

from visualization_utils import fmt_est_ci, make_pivot_with_ci

race_order = ["Black/African American", "Hispanic/Latino", "White", "Asian", "Other"]

df = policing_analysis.copy()

pop = (df[["Perceived Race", "Population"]]
       .drop_duplicates("Perceived Race")
       .set_index("Perceived Race")
       .reindex(race_order))

stops_1k = make_pivot_with_ci(df, "Stops per 1,000", "Stops per 1,000 SE")
searches_1k = make_pivot_with_ci(df, "Searches per 1,000", "Searches per 1,000 SE")

In [ ]:
table1 = pd.concat(
    [pop, stops_1k, searches_1k],
    axis=1,
    keys=["Population", "Stops per 1,000", "Searches per 1,000"]
)


table1.columns = pd.MultiIndex.from_tuples(
    [(g, c) for g, c in table1.columns],
    names=["", "Year"]
)

table1

In [ ]:
tab2 = df.copy()

# Convert proportions to percentages
tab2["Search Rate"] *= 100
tab2["Hit Rate"] *= 100
tab2["Search Rate SE"] *= 100
tab2["Hit Rate SE"] *= 100

search_rate = make_pivot_with_ci(tab2, "Search Rate", "Search Rate SE")
hit_rate    = make_pivot_with_ci(tab2, "Hit Rate", "Hit Rate SE")

table2 = pd.concat([search_rate, hit_rate], axis=1, keys=["Search Rate (%)", "Hit Rate (%)"])

table2.columns = pd.MultiIndex.from_tuples(
    [(g, c) for g, c in table2.columns],
    names=["", "Year"]
)

table2 = table2.reindex(race_order)
table2

In [ ]:
from visualization_utils import(visualize_policing, export_figures_to_pdf)  # noqa: F401

policing_figs = visualize_policing(policing_analysis)
# export_figures_to_pdf(policing_figs, output_dir='../output')

# Policing Sensitivity Analysis: Mixed Search Bases

In [ ]:
from rates_utils import policing_rates_sensitivity

# Create alternative classification: treat mixed searches as discretionary
policing["disc_search_mixed"] = policing["action_any_search"] & policing["search_type"].isin(["Discretionary only", "Mixed", "No search basis"])

# Create discretionary search indicator
policing["disc_hit_mixed"] = (
    policing["disc_search_mixed"] & # Cannot be from a non-discretionary search
    policing["action_any_search"].fillna(False) & # Has to have been searched
    (policing["contraband_any"] == 1)
)

# Compute policing rates using mixed classification
policing_analysis_mixed = policing_rates_sensitivity(policing, census_coarse, "mixed")

In [ ]:
def make_2024_block(df, label):
    """Return a 2024 table with formatted Search/Hit rates for one method."""
    d = df[df["Year"] == 2024].copy()

    out = pd.DataFrame(index=d["Perceived Race"])
    out[( "Search Rate (%)", label)] = [fmt_est_ci(e, s, 2) for e, s in zip(d["Search Rate"], d["Search Rate SE"])]
    out[( "Hit Rate (%)",    label)] = [fmt_est_ci(e, s, 2) for e, s in zip(d["Hit Rate"],    d["Hit Rate SE"])]
    return out

# Make safe copies
strict_df = policing_analysis.copy()
mixed_df  = policing_analysis_mixed.copy()

# Convert to percentages ONLY if stored as proportions
for d in (strict_df, mixed_df):
    if d["Search Rate"].max() <= 1:
        d[["Search Rate", "Hit Rate",
           "Search Rate SE", "Hit Rate SE"]] *= 100

strict_2024 = make_2024_block(strict_df, "Strict")
mixed_2024  = make_2024_block(mixed_df, "Mixed")

table3 = strict_2024.join(mixed_2024, how="outer")

# Enforce race order
race_order = ["Black/African American", "Hispanic/Latino", "White", "Asian", "Other"]
table3 = table3.reindex(race_order)

table3 = table3[[("Search Rate (%)","Strict"), ("Search Rate (%)","Mixed"),
                 ("Hit Rate (%)","Strict"),    ("Hit Rate (%)","Mixed")]]

table3.columns = pd.MultiIndex.from_tuples(table3.columns, names=["", ""])
table3.index.name = "Perceived Race"

table3

In [ ]:
from visualization_utils import create_sensitivity_visualization

# Create the sensitivity analysis figure
sensitivity_fig = create_sensitivity_visualization(
    strict_df=policing_analysis,
    mixed_df=policing_analysis_mixed,
    strict_name="Strict",
    mixed_name="Mixed",
    title_ending="Search Classification"
)

# export_figures_to_pdf({"sensitivity_analysis": sensitivity_fig}, output_dir='../output')

# Policing Sensitivity Analysis: Multi-Person Stops

In [ ]:
# Rename column
policing_multi.rename(columns={"disc_search": "disc_search_multiperson"}, inplace=True)

# Create discretionary search indicator
policing_multi["disc_hit_multiperson"] = (
    policing_multi["disc_search_multiperson"] & # Cannot be from a non-discretionary search
    policing_multi["action_any_search"].fillna(False) & # Has to have been searched
    (policing_multi["contraband_any"] == 1)
)

# Compute policing rates using all people involved in stops, not just the first person listed
policing_analysis_multiperson = policing_rates_sensitivity(policing_multi, census_coarse, "multiperson")

# Make safe copies
solo_df = policing_analysis.copy()
multiperson_df  = policing_analysis_multiperson.copy()

# Convert to percentages ONLY if stored as proportions
for d in (solo_df, multiperson_df):
    if d["Search Rate"].max() <= 1:
        d[["Search Rate", "Hit Rate",
           "Search Rate SE", "Hit Rate SE"]] *= 100

solo_2024 = make_2024_block(solo_df, "Solo")
multiperson_2024  = make_2024_block(multiperson_df, "Multiperson")

table4 = solo_2024.join(multiperson_2024, how="outer")

# Enforce race order
race_order = ["Black/African American", "Hispanic/Latino", "White", "Asian", "Other"]
table4 = table4.reindex(race_order)

table4 = table4[[("Search Rate (%)","Solo"), ("Search Rate (%)","Multiperson"),
                 ("Hit Rate (%)","Solo"),    ("Hit Rate (%)","Multiperson")]]

table4.columns = pd.MultiIndex.from_tuples(table4.columns, names=["", ""])
table4.index.name = "Perceived Race"

table4

In [ ]:
# Create the sensitivity analysis figure
sensitivity_multiperson_fig = create_sensitivity_visualization(
    policing_analysis,
    policing_analysis_multiperson,
    strict_name="Solo",
    mixed_name="Multiperson",
    title_ending="Stops"
)

# export_figures_to_pdf({"sensitivity_analysis": sensitivity_fig}, output_dir='../output')

# Prosecution Analysis: Enhancement Rates by Charge and Statute Level

In [ ]:
from rates_utils import categorize_charge, enhancement_rates_by_primary_severity

# Harmonizes race labels
prosecution = add_standardized_race(prosecution, 'canonical_race')

# Identify charges that are enhancements/priors/sentencing considerations rather than actual substantive criminal charges
prosecution['is_non_substantive'] = (
    prosecution['is_enhancement_charge'].astype(bool) |
    prosecution['is_special_circumstance'].astype(bool) |
    prosecution['is_sentencing_charge'].astype(bool)
)

# Apply categorization ONLY to substantive charges
prosecution.loc[~prosecution['is_non_substantive'], 'charge_category'] = \
    prosecution.loc[~prosecution['is_non_substantive'], 'charge_description'].apply(categorize_charge)

# For non-substantive charges, label them explicitly as ehancement/prior/sentencing
prosecution.loc[prosecution['is_non_substantive'], 'charge_category'] = 'Enhancement/Prior/Sentencing'

# Each case is a single defendant, so case-level = defendant-case-level
# For each case, if any charge is an enhancement charge, then the case is flagged as having an enhancement charge
enh_flag = (
    prosecution.groupby('source_case_id')['is_enhancement_charge']
    .max()
    .rename('any_enhancement_in_case')
    .reset_index()
)

# Drop the enhancement rows, keeping only the base charges
substantive = prosecution[~prosecution['is_non_substantive']].copy()

# Add the enhancement flag to the dataframe
substantive = substantive.merge(enh_flag, on='source_case_id', how='left')
substantive['any_enhancement_in_case'] = substantive['any_enhancement_in_case'].fillna(0).astype(int)

# Calculate enhancement rates by charge category
# For each case, a primary charge category and statute level is assigned
# Then computes cases with any enhancement charge over total cases for each charge category
enhancement_by_primary = enhancement_rates_by_primary_severity(substantive)

# Calculate and display average enhancement rates by primary charge category
enhancement_by_primary.groupby("primary_charge_category").apply(lambda g: g["Enhanced"].sum() / g["N"].sum()).sort_values(ascending=False)

# Exclude any categories with less than a 5% overall enhancement rate to focus on the most relevant categories
relevant_categories = [
    "DUI",
    "Assault/Violence",
    "Weapons"
]
relevant_enhancements = enhancement_by_primary[enhancement_by_primary["primary_charge_category"].isin(relevant_categories)]

In [ ]:
# Enhancement rate tables

from visualization_utils import (
    enhancement_rate_race_table,
    enhancement_rate_race_statute_table,
    enhancement_rate_race_statute_category_table
)

# Table 1: Overall by race
overall_by_race = enhancement_rate_race_table(enhancement_by_primary)
display(overall_by_race)

# Table 2: By race and statute level
by_race_statute = enhancement_rate_race_statute_table(enhancement_by_primary)
display(by_race_statute)

# Table 3: By race, statute level, and primary category (for all categories)
by_all_three = enhancement_rate_race_statute_category_table(
    enhancement_by_primary, 
    top_categories=relevant_categories
)
display(by_all_three)

In [ ]:
from visualization_utils import (
    plot_enhancement_rate_by_race,
    plot_enhancement_rate_by_race_statute,
    plot_enhancement_rate_by_race_statute_category
)

# Not stratifying by year because samples sizes get too small

# Exclude "Other" in plots
enhancement_no_other = enhancement_by_primary[enhancement_by_primary["race_std"] != "Other"].copy()
relevant_enhancements_no_other = relevant_enhancements[relevant_enhancements["race_std"] != "Other"].copy()

# Plot 1: Overall by race
enhancement_rate_race = plot_enhancement_rate_by_race(enhancement_no_other)

# Plot 2: By race and statute level
enhancement_rate_race_statute = plot_enhancement_rate_by_race_statute(enhancement_no_other)

# Plot 3: By race, statute level, and categories (for only categories with at least 5% overall enhancement rate)
enhancement_rate_race_statute_category = plot_enhancement_rate_by_race_statute_category(relevant_enhancements_no_other)

# Prosecution Analysis: Wobblers

In [ ]:
from visualization_utils import wobbler_felony_rate_tables

wobbler_table, wobbler_category_table = wobbler_felony_rate_tables(substantive)

display(wobbler_table)
display(wobbler_category_table)

In [ ]:
#  Prepare data for plotting (exclude "Other" race)
plot_data = wobbler_by_category.reset_index()
plot_data = plot_data[plot_data['race_std'] != 'Other'].copy()

# Calculate 95% confidence intervals
plot_data['ci_lower'] = (plot_data['Felony Rate'] - 1.96 * plot_data['Felony Rate SE']) * 100
plot_data['ci_upper'] = (plot_data['Felony Rate'] + 1.96 * plot_data['Felony Rate SE']) * 100
plot_data['Felony Rate %'] = plot_data['Felony Rate'] * 100

# Define color palette
race_colors = {
    'Black/African American': '#e74c3c',
    'Hispanic/Latino': '#3498db',
    'White': '#2ecc71',
    'Asian': '#f39c12'
}

# Get top charge categories by total volume
top_categories = (
    plot_data
    .groupby('primary_charge_category')['Total']
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)

# Filter to top categories
plot_data_filtered = plot_data[plot_data['primary_charge_category'].isin(top_categories)]

# Create the plot
wobbler_stratified_fig = (
    ggplot(plot_data_filtered, 
           aes(x='reorder(primary_charge_category, -`Felony Rate %`)', 
               y='`Felony Rate %`', 
               fill='race_std'))
    + geom_col(position='dodge', width=0.7)
    + geom_errorbar(
        aes(ymin='ci_lower', ymax='ci_upper', group='race_std'),
        position=position_dodge(0.7),
        width=0.25,
        color='black',
        size=0.5
    )
    + scale_fill_manual(values=race_colors)
    + labs(
        title='Wobbler Felony Filing Rates by Race and Primary Charge Category',
        subtitle='Top 5 charge categories by volume',
        x='Primary Charge Category',
        y='Felony Filing Rate (%)',
        fill='Race/Ethnicity'
    )
    + theme_minimal()
    + theme(
        figure_size=(14, 8),
        axis_text_x=element_text(angle=45, ha='right'),
        plot_title=element_text(size=14, weight='bold'),
        plot_subtitle=element_text(size=11),
        legend_position='right'
    )
)

# Display the figure
wobbler_stratified_fig

# Optional: Export to PDF
# from visualization_utils import export_figures_to_pdf
# export_figures_to_pdf({'wobbler_stratified': wobbler_stratified_fig}, output_dir='../output')


In [ ]:
from visualization_utils import plot_wobbler_felony_rates

# Create the wobbler visualization using the harmonized function
wobbler_fig = plot_wobbler_felony_rates(wobbler_summary)
# export_figures_to_pdf({'wobbler_analysis': wobbler_fig}, output_dir='../output')